In [ ]:
import pandas as pd
import joblib
import optuna
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.feature_selection import SelectKBest, f_classif
from imblearn.over_sampling import SMOTE
import seaborn as sns
import matplotlib.pyplot as plt



In [ ]:
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
df = pd.read_csv("../data/cardiovascular.csv")

print("Shape:", df.shape)
df.head()


In [ ]:
# Class distribution
plt.figure()
sns.countplot(x=df['risk_category'])
plt.title("Diabetes Class Distribution")
plt.show()


In [ ]:
print("\nMissing Values:\n", df.isnull().sum())
print("\nTarget Values:\n", df["risk_category"].unique())


In [ ]:
df.drop(columns=["Patient_ID", "heart_disease_risk_score"], inplace=True)
df.dropna(inplace=True)
# Clean target
df["risk_category"] = df["risk_category"].astype(str).str.strip().str.lower()

mapping = {
    "low": 0,
    "medium": 1,
    "high": 1
}

df["risk_category"] = df["risk_category"].map(mapping)

# Remove invalid rows
df = df.dropna(subset=["risk_category"])

print("\nAfter Mapping:\n", df["risk_category"].value_counts())


In [ ]:
# Prepare features and target
X = df.drop("risk_category", axis=1)
y = df["risk_category"]

# Train/validation/test split (60/20/20)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)


In [ ]:
# Identify numeric and categorical columns
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {numeric_cols}")
print(f"Categorical columns: {categorical_cols}")

# Create preprocessor: StandardScaler for numeric, OneHotEncoder for categorical
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
])

# Fit and transform training data
X_train_prep = preprocessor.fit_transform(X_train)
X_val_prep = preprocessor.transform(X_val)
X_test_prep = preprocessor.transform(X_test)

# Get feature names after transformation
feature_names_after = numeric_cols + list(preprocessor.named_transformers_['cat'].get_feature_names_out(categorical_cols))
print(f"Features after preprocessing: {len(feature_names_after)}")
print(f"Split sizes -> train: {X_train.shape}, val: {X_val.shape}, test: {X_test.shape}")

In [ ]:
smote = SMOTE(random_state=42)
X_train_prep, y_train = smote.fit_resample(X_train_prep, y_train)
print("SMOTE applied - training data balanced")

In [ ]:
# FEATURE SELECTION

print("\nFinding Best K Value...")

k_values = [5, 8, 10, 12, 15]

best_k = None
best_score = 0

for k in k_values:

    temp_selector = SelectKBest(f_classif, k=k)

    X_train_temp = temp_selector.fit_transform(X_train_prep, y_train)
    X_val_temp = temp_selector.transform(X_val_prep)

    temp_model = RandomForestClassifier(random_state=42)

    temp_model.fit(X_train_temp, y_train)

    preds = temp_model.predict(X_val_temp)

    score = f1_score(y_val, preds)

    print(f"K = {k} --> F1 Score = {score:.4f}")

    if score > best_score:
        best_score = score
        best_k = k

print(f"\nBest K Selected: {best_k}")

# FINAL SELECTKBEST
selector = SelectKBest(f_classif, k=best_k)

X_train_prep = selector.fit_transform(X_train_prep, y_train)
X_val_prep = selector.transform(X_val_prep)
X_test_prep = selector.transform(X_test_prep)

selected_idx = selector.get_support(indices=True)

selected_features = [
    feature_names_after[i]
    for i in selected_idx
]

print("\nFeature Selection Complete:")
print(f"Selected {len(selected_features)} features:")
print(selected_features)

In [ ]:
# Baseline Model Training
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1),
    "SVM": SVC(probability=True, random_state=42)
}

baseline_results = []
for name, model in models.items():
    model.fit(X_train_prep, y_train)
    y_val_pred = model.predict(X_val_prep)

    baseline_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1": f1_score(y_val, y_val_pred),
    })

baseline_results_df = pd.DataFrame(baseline_results)[["Model", "Accuracy", "Precision", "Recall", "F1"]]
print("\n-- Baseline Model Comparison (Validation) --")
display(baseline_results_df.sort_values("F1", ascending=False).reset_index(drop=True))


In [ ]:
# Optuna Setup
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


In [ ]:
# Optuna Optimization - Logistic Regression

def objective_lr(trial):
    params = {
        "C": trial.suggest_float("C", 1e-4, 1e2, log=True),
        "solver": trial.suggest_categorical("solver", ["liblinear", "lbfgs", "saga"]),
        "max_iter": 1000,
        "random_state": 42,
    }
    model = LogisticRegression(**params)
    scores = cross_val_score(model, X_train_prep, y_train, cv=cv, scoring="f1", n_jobs=-1)
    return float(scores.mean())

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_lr = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_lr.optimize(objective_lr, n_trials=20)

optimized_lr = LogisticRegression(**study_lr.best_params, max_iter=1000, random_state=42)
optimized_lr.fit(X_train_prep, y_train)


In [ ]:
# Optuna Optimization - Random Forest

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "random_state": 42,
        "n_jobs": -1,
    }
    model = RandomForestClassifier(**params)
    scores = cross_val_score(model, X_train_prep, y_train, cv=cv, scoring="f1", n_jobs=-1)
    return float(scores.mean())

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_rf.optimize(objective_rf, n_trials=20)

optimized_rf = RandomForestClassifier(**study_rf.best_params, random_state=42, n_jobs=-1)
optimized_rf.fit(X_train_prep, y_train)


In [ ]:
# Optuna Optimization - XGBoost

def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 2, 12),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "random_state": 42,
        "eval_metric": "logloss",
        "n_jobs": -1,
    }
    model = XGBClassifier(**params)
    scores = cross_val_score(model, X_train_prep, y_train, cv=cv, scoring="f1", n_jobs=-1)
    return float(scores.mean())

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_xgb = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=30)

optimized_xgb = XGBClassifier(**study_xgb.best_params, random_state=42, eval_metric="logloss", n_jobs=-1)
optimized_xgb.fit(X_train_prep, y_train)


In [ ]:
# Optuna Optimization - SVM

def objective_svm(trial):
    kernel = trial.suggest_categorical("kernel", ["linear", "rbf"])
    params = {
        "C": trial.suggest_float("C", 1e-4, 1e2, log=True),
        "kernel": kernel,
        "probability": True,
        "random_state": 42,
    }
    if kernel == "rbf":
        params["gamma"] = trial.suggest_float("gamma", 1e-4, 1e0, log=True)

    model = SVC(**params)
    scores = cross_val_score(model, X_train_prep, y_train, cv=cv, scoring="f1", n_jobs=-1)
    return float(scores.mean())

optuna.logging.set_verbosity(optuna.logging.WARNING)
study_svm = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study_svm.optimize(objective_svm, n_trials=30)

optimized_svm = SVC(**study_svm.best_params, probability=True, random_state=42)
optimized_svm.fit(X_train_prep, y_train)


In [ ]:
# Optimized Model Comparison
optimized_models = {
    "Logistic Regression": optimized_lr,
    "Random Forest": optimized_rf,
    "XGBoost": optimized_xgb,
    "SVM": optimized_svm,
}

optimized_results = []
for name, model in optimized_models.items():
    y_val_pred = model.predict(X_val_prep)

    optimized_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_val, y_val_pred),
        "Precision": precision_score(y_val, y_val_pred),
        "Recall": recall_score(y_val, y_val_pred),
        "F1": f1_score(y_val, y_val_pred),
    })

optimized_results_df = pd.DataFrame(optimized_results)[["Model", "Accuracy", "Precision", "Recall", "F1"]]
print("\n-- Optimized Model Comparison (Validation) --")
display(optimized_results_df.sort_values("F1", ascending=False).reset_index(drop=True))


In [ ]:
# Best Optimized Model Selection
best_row = optimized_results_df.sort_values("F1", ascending=False).iloc[0]
best_model_name = best_row["Model"]
best_model = optimized_models[best_model_name]
print(f"\nBest Optimized Model: {best_model_name}")
print(f"Validation F1: {best_row['F1']:.4f}")


In [ ]:
# Threshold Tuning
threshold_candidates = [round(i * 0.05, 2) for i in range(6, 15)]
val_probs = best_model.predict_proba(X_val_prep)[:, 1]

best_threshold = 0.5
best_threshold_f1 = -1.0
for t in threshold_candidates:
    score = f1_score(y_val, (val_probs >= t).astype(int))
    if score > best_threshold_f1:
        best_threshold_f1 = score
        best_threshold = t

threshold = float(best_threshold)
print(f"Best threshold: {threshold}")
print(f"Validation F1 at best threshold: {best_threshold_f1:.4f}")


In [ ]:
# Final Evaluation
y_test_prob = best_model.predict_proba(X_test_prep)[:, 1]
y_test_pred = (y_test_prob >= threshold).astype(int)
cm_test = confusion_matrix(y_test, y_test_pred)

print(f"\n-- Final Test Performance [{best_model_name} | threshold={threshold}] --")
print(f"Accuracy  : {accuracy_score(y_test, y_test_pred):.4f}")
print(f"Precision : {precision_score(y_test, y_test_pred):.4f}")
print(f"Recall    : {recall_score(y_test, y_test_pred):.4f}")
print(f"F1 Score  : {f1_score(y_test, y_test_pred):.4f}")

plt.figure(figsize=(6, 4))
sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
# Artifact Saving
final_artifact = {
    "model": best_model,
    "preprocessor": preprocessor,
    "selector": selector,
    "selected_features": selected_features,
    "feature_columns": list(X.columns),
    "threshold": threshold,
    "model_name": best_model_name,
    "best_model_name": best_model_name,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
}

joblib.dump(final_artifact, "../models/cardiovascular_model.pkl")

print("\nCardiovascular model saved successfully!")
print(f"Model: {best_model_name}")
print(f"Threshold: {threshold}")
print(f"Selector: SelectKBest(k={best_k})")
print("Preprocessor: StandardScaler + OneHotEncoder")


In [ ]:
artifact = joblib.load("../models/cardiovascular_model.pkl")

model = artifact["model"]
best_model_name = artifact.get("best_model_name", artifact.get("model_name", "Trained Model"))
preprocessor = artifact["preprocessor"]
selector = artifact["selector"]
threshold = artifact["threshold"]
feature_columns = artifact.get("feature_columns", artifact.get("features"))

# Reusable probability-to-severity mapping

def get_risk_level(probability):
    prob_percent = probability * 100
    if prob_percent <= 30:
        return "Low Risk"
    if prob_percent <= 60:
        return "Moderate Risk"
    if prob_percent <= 80:
        return "High Risk"
    return "Critical Risk"

data = {
    "age": 52,
    "bmi": 28.7,
    "systolic_bp": 132,
    "diastolic_bp": 84,
    "cholesterol_mg_dl": 210,
    "resting_heart_rate": 78,
    "smoking_status": "Current",
    "daily_steps": 6200,
    "stress_level": 6,
    "physical_activity_hours_per_week": 3.0,
    "sleep_hours": 6.5,
    "family_history_heart_disease": "Yes",
    "diet_quality_score": 58,
    "alcohol_units_per_week": 4.0
}

row = pd.DataFrame([data])

# Apply the same preprocessing stack used during training
X_new = preprocessor.transform(row)
X_new = selector.transform(X_new)

prob = model.predict_proba(X_new)[0, 1]
pred = 1 if prob >= threshold else 0

print("Risk:", "High" if pred == 1 else "Low")
print("Risk Level:", get_risk_level(prob))
print("Probability:", round(prob, 3))
print("Model:", best_model_name)
print("Threshold:", threshold)
